In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
print("google.colab" in sys.modules)

True


In [ ]:
import os

os.listdir('/content/drive/MyDrive')

['Colab Notebooks', 'cnn-project']

In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer,TrainingArguments , T5ForConditionalGeneration



In [ ]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [ ]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [ ]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [ ]:
train_data.sample(15)

,id,dialogue,summary
4742,13811908,Violet: hi! i came across this Austin's articl...,Violet sent Claire Austin's article.
8870,13716431,Pat: So does anyone know when the stream is go...,Pat and Lou are waiting for The stream but Kev...
6554,13810214,Jane: <gif_file>\r\nJane: Whaddya think? \r\nS...,Jane is updating her Tinder profile tonight an...
12900,13729823,"Adam: Do u have a map of Paris?\r\nTom: Yes, W...",Tom has a map of Paris.
2596,13681400,"Frank: Hi, how's the family?\r\nMike: great! S...","Mike is happy, because Sam's moved out. Mike a..."
6422,13716070,Paul: Lucky you!\r\nJohn: ?\r\nPete: Our class...,"John, Pete and Paul's classes have been cancel..."
2452,13727976,Jasper: i miss you so much already :(\r\nKaren...,Karen will be back on Sunday. Karen and Jasper...
476,13681231,Ken: how long do you need?\r\nJude: i think ab...,Ken will wait inside as Jude needs 10 more min...
14716,13862652,"Victoria: Hey, I am in the toilet...And..\nSky...",Victoria is in a restaurant toilet and texts S...
10143,13728508,Sandra: Do u need any help with the party tomo...,Ronda does not need any help with the party to...


In [ ]:
train_data.shape

(14732, 3)

In [ ]:
val_data.shape

(818, 3)

In [ ]:
# random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [ ]:
# Data pre-processing

import re #regular expressions

def clean_data(text):
  text = re.sub(r"\r\n"," ",text) #lines
  text = re.sub(r"\s+"," ",text) #spaces
  text = re.sub(r"<.*?>"," ",text) # html tags
  text = text.strip().lower()
  return text


In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)


In [ ]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

In [ ]:
# Tokenization
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [ ]:

def tokenize(data):

    inputs = tokenizer(
        data["dialogue"],
        padding="max_length",
        max_length=512,
        truncation=True
    )

    targets = tokenizer(
        data["summary"],
        padding="max_length",
        max_length=150,
        truncation=True
    )

    inputs["labels"] = targets["input_ids"]

    return inputs


In [ ]:
train_dataset = train_data.apply(tokenize,axis=1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()



In [ ]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
# input ids - dialogue => token_ids
# 1 => eos ending of sequence

#attention masks

#labels-target => summary token

In [ ]:
len(train_dataset[0]["input_ids"])

512

In [ ]:
len(train_dataset[0]["labels"])

150

In [ ]:
type(train_dataset)

list

In [ ]:
type(val_dataset)

list

In [ ]:
# working with model

model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
# fine-tune
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model.to(device)


device: cuda
GPU: Tesla T4


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
#training arguments

training_args = TrainingArguments(
    output_dir= "./results",
    num_train_epochs=3,
    weight_decay=0.01,
    per_device_train_batch_size=3,
    per_device_eval_batch_size=3,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=400


)

In [ ]:
trainer = Trainer(
    model = model,
    args =training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.404546,0.360445
2,0.372988,0.351366
3,0.362351,0.350496


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4002, training_loss=0.7617326196523263, metrics={'train_runtime': 685.0007, 'train_samples_per_second': 17.518, 'train_steps_per_second': 5.842, 'total_flos': 1624101617664000.0, 'train_loss': 0.7617326196523263, 'epoch': 3.0})

In [ ]:
#saving the model

model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary model/tokenizer_config.json',
 './saved_summary model/tokenizer.json')

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)

    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    )

    # Move inputs to GPU/CPU
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.to(device)
    model.eval()

    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(
        targets[0],
        skip_special_tokens=True
    )

    return summary

In [ ]:
test_dialogue = """Customer: Hi, I ordered a laptop last week but it still hasn't arrived.

Agent: I'm sorry to hear that. Can you provide your order number?

Customer: Yes, it's ORD-45892.

Agent: Thank you. Let me check the status.

Agent: I can see the package was delayed due to weather conditions and is expected to arrive in two days.

Customer: Okay, thanks for the update. Can I get a notification when it ships?

Agent: Yes, I've enabled SMS and email notifications for your order.

Customer: Great, thank you.

Agent: You're welcome. Is there anything else I can help you with today?

Customer: No, that's all."""

In [ ]:
summary = summarize_dialogue(test_dialogue)
print("summary: ",summary)

summary:  customer ordered a laptop last week but it still hasn't arrived. agent has enabled sms and email notifications for the order.
